# 83장 · 멱등적인 완료 상태 갱신

멱등성은 같은 요청을 여러 번 적용해도 최종 상태가 같다는 성질입니다. 완료 여부를 뒤집는 함수와 원하는 값을 지정하는 함수를 비교합니다.

완료 요청을 두 번 보냈을 때 체크 표시가 풀리는 오류를 재현합니다. 현재 상태를 뒤집는 구현과 요청의 completed 값을 대입하는 구현을 같은 입력으로 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 토글 방식과 중복 요청 재현

제공된 faulty 저장소는 completed=True 요청을 ‘현재 값을 뒤집기’로 처리합니다. 같은 ID, 같은 초기 상태, 같은 입력을 사용해야 수정 전후 결과를 비교할 수 있습니다.

**같은 완료 요청의 반복과 멱등성**　완료 요청을 두 번 보냈을 때 토글 구현의 최종값은 False, 목표값을 대입한 구현은 True입니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter83/index.html#practice-space)


### 동일한 요청을 두 번 보내기

<p>결함을 보려면 같은 딕셔너리에 목표 True를 두 번 전달해야 합니다. 호출할 때마다 새 딕셔너리를 만들면 첫 번째 요청만 반복하는 셈이어서 두 번째 호출의 문제를 발견할 수 없습니다.</p>


### 셀 01 · 반복 요청으로 토글 오류 재현

아래 결함은 수업용으로 의도해 넣었습니다. 두 번째 결과를 첫 번째와 비교하세요.


In [1]:
def faulty_update(item, completed):
    item["completed"] = not item["completed"]
    return item["completed"]

item = {"id": 1, "title": "복습", "completed": False}
print(faulty_update(item, True))
print(faulty_update(item, True))

True
False


결함 보고에 두 요청의 completed 인자와 각 실행 직후의 상태를 함께 기록합니다.

완료 요청을 두 번 보냈더니 True 뒤에 False가 나옵니다. 네트워크 재시도로 같은 요청이 반복되면 사용자의 목표와 반대로 끝나는 구현입니다. 목표값 True를 대입하는 방식은 두 요청 뒤에도 완료 상태가 유지되며, 완료 해제는 False를 보내 별도로 검사합니다.

<div class="example-table"><table><caption>동일한 완료 요청을 두 번 보낸 결과</caption><thead><tr><th scope="col">수정 방식</th><th scope="col">초기 상태</th><th scope="col">첫 요청</th><th scope="col">재전송</th></tr></thead><tbody><tr><td>현재 값 반전</td><td>False</td><td>True</td><td>False</td></tr><tr><td>목표값 대입</td><td>False</td><td>True</td><td>True</td></tr></tbody></table></div>


### 호출 횟수에 따라 바뀌는 토글

<p>faulty_update는 전달된 completed 인자를 사용하지 않습니다. 현재 값에 not을 적용하므로 이미 완료된 행도 같은 요청을 한 번 더 받으면 미완료가 됩니다.</p>


## 목표 상태 대입과 회귀 검사

수정 범위를 완료 필드 반영 한 줄에 둡니다. PATCH 요청이 들어온 횟수에 관계없이 명시적으로 전달받은 상태를 저장해야 합니다.


### 과제 02 · 목표 상태 대입의 멱등성

원래 함수에서 받은 completed 값을 그대로 대입하고 같은 입력 두 번의 결과를 출력하세요.


In [2]:
def update(item, completed):
    # 요청에 담긴 completed를 반영하세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def update(item, completed):
    item[&#x27;completed&#x27;] = completed
    return item[&#x27;completed&#x27;]

row = {&#x27;id&#x27;: 1, &#x27;title&#x27;: &#x27;복습&#x27;, &#x27;completed&#x27;: False}
print(update(row, True), update(row, True))</code></pre>
<p>실행 결과</p><pre><code>True True</code></pre>
<p>True True가 나옵니다. 두 번째 요청을 삭제해서 문제를 숨기지 않고 입력을 동일하게 유지했습니다.</p>
</details>


### 목표 상태의 직접 대입

<p>수정한 함수는 호출자가 원하는 값을 item["completed"]에 대입해야 합니다. 같은 True를 여러 번 대입해도 저장값은 True로 유지되고, False를 전달한 경우에만 완료가 해제됩니다.</p>


#### 동일 입력 반복에 따른 최종 상태

<table style="width:100%;border-collapse:collapse"><thead><tr><th scope="col">구현</th><th scope="col">첫 요청 True</th><th scope="col">둘째 요청 True</th></tr></thead><tbody><tr><td>셀 01 · 현재 값을 뒤집음</td><td>True</td><td>False</td></tr><tr><td>과제 02 · 요청 값을 저장</td><td>True</td><td>True</td></tr></tbody></table>


### 셀 03 · 완료 해제의 회귀 테스트

과제 02에서 완성한 update에 True, True, False를 보내고 매번 반환값과 같은 행의 저장값을 확인합니다.

과제 02에서 작성한 update를 실행한 뒤 이 셀로 오세요. 같은 True 요청을 두 번 보내고 이어 False로 되돌립니다. 셀을 고쳤다면 함수 정의 셀부터 다시 실행하세요.


In [3]:
row = {"id": 5, "title": "복습", "completed": False}
first = update(row, True)
after_first = row.copy()
second = update(row, True)
after_second = row.copy()
last = update(row, False)
if first is None and second is None and last is None:
    print("과제 02의 update를 완성하고 정의 셀부터 다시 실행하세요.")
else:
    assert first is True and second is True, "같은 완료 요청을 반복해도 True를 반환해야 합니다."
    expected = {"id": 5, "title": "복습", "completed": True}
    assert after_first == expected and after_second == expected, "두 완료 요청 모두 원래 행에 저장되어야 합니다."
    assert last is False, "완료 해제 요청은 False를 반환해야 합니다."
    assert row == {"id": 5, "title": "복습", "completed": False}, "같은 행의 ID·제목·완료 값을 확인하세요."
    print("검사 통과", row)

과제 02의 update를 완성하고 정의 셀부터 다시 실행하세요.


시작본에서 과제 02를 비워 두면 update를 완성하라는 안내가 나옵니다. 정답을 넣고 과제 02와 이 셀을 다시 실행했을 때의 마지막 줄은 다음과 같습니다.

검사 통과 {'id': 5, 'title': '복습', 'completed': False}

완료 요청을 두 번 보낸 뒤 해제했으므로 마지막 값은 False입니다. 요청할 때마다 상태를 뒤집는 구현은 두 번째 True 검사에서 멈춥니다.


### 반환값과 실제 저장값을 따로 검사하기

<p>first와 second의 반환값이 모두 True인지 보고, 두 시점에 복사한 행의 id·title·completed도 대조합니다. True를 반환하기만 하고 딕셔너리를 고치지 않은 함수라면 행 비교에서 드러납니다.</p>


## 돌아보기

수정 요청의 입력·기대·실제를 다른 팀원에게 보여 주고 초기 상태부터 재현하게 해 보세요. 같은 결과를 보지 못한다면 게시된 코드를 더 바꾸기 전에 사용한 저장소와 초기 데이터를 비교하세요.
